In [ ]:
import pandas as pd
import sqlite3
import time
import matplotlib.pyplot as plt

df = pd.read_csv("..\\data\\arrhythmia.csv", header=None, na_values="?")

print(df.shape)
print(df.head())

In [ ]:
def clean_data(data):
    # Work on a copy so the supplied DataFrame is not changed
    clean = data.copy(deep=True)

    # Standardise sex labels if letter labels are present
    clean["sex"] = clean["sex"].replace({
        "M": 0,
        "F": 1,
        "m": 0,
        "f": 1
    })

    # Calculate BMI to check whether height and weight
    # are internally consistent
    clean["bmi"] = (
        clean["weight"] /
        (clean["height"] / 100) ** 2
    )

    # Identify clearly unreliable demographic records
    invalid_record = (
        # A height above 250 cm is physically implausible
        (clean["height"] > 250)

        # A recorded height above 100 cm for age 0 or 1
        # is highly inconsistent with the recorded age
        | ((clean["age"] <= 1) &
            (clean["height"] > 100)
        )

        # An adult BMI below 10 suggests that the recorded
        # height and weight are internally inconsistent
        | (
            (clean["age"] >= 20) &
            (clean["bmi"] < 10)
        )
    )

    # Display the records before removing them
    print("Clearly unreliable records removed during cleaning:")
    print(
        clean.loc[
            invalid_record,
            [
                "age",
                "sex",
                "height",
                "weight",
                "bmi",
                "class"
            ]
        ].round(2))

    # Remove the clearly unreliable records
    clean = clean.loc[~invalid_record].copy()

    # Recalculate BMI for the retained records
    clean["bmi"] = (
        clean["weight"] /
        (clean["height"] / 100) ** 2
    )

    # Fill ECG columns containing only a small
    # number of missing values
    for col in [
        "p_angle",
        "t_angle",
        "qrst_angle",
        "heart_rate"
    ]:
        clean[col] = clean[col].fillna(
            clean[col].median()
        )

    # Drop j_angle because most of its values are missing
    clean = clean.drop(
        columns=["j_angle"],
        errors="ignore")

    # Remove exact duplicate rows
    clean = clean.drop_duplicates()

    # Create a simpler yes/no diagnosis column
    clean["arrhythmia_present"] = (
        clean["class"] != 1
    )

    # Reset the row numbers after records are removed
    clean = clean.reset_index(drop=True)

    return clean

In [ ]:
df.columns = ["age","sex","height","weight","qrs_duration","pr_interval",
              "qt_interval","t_interval","p_interval","qrs_angle","t_angle",
              "p_angle","qrst_angle","j_angle","heart_rate"] \
             + list(df.columns[15:-1]) + ["class"]
print(df.columns[-3:])
print(df.head())

# Keep an untouched copy of the loaded and named dataset
df_original = df.copy(deep=True)

In [ ]:
# Check missing values
missing = df_original.isna().sum()
print("Columns containing missing values:")
print(missing[missing > 0].sort_values(ascending=False))

# Check duplicate rows
print("\nNumber of duplicate rows:")
print(df_original.duplicated().sum())

# Check the ranges of important measurements
range_cols = ["age", "height", "weight", "heart_rate"]
print("\nMinimum and maximum values:")
print(df_original[range_cols].agg(["min", "max"]))

In [ ]:
review_cols = [
    "age",
    "sex",
    "height",
    "weight",
    "class"
]
records_to_review = df_original.loc[
    (df_original["age"] < 18) |
    (df_original["height"] < 120) |
    (df_original["height"] > 220) |
    (df_original["weight"] < 25),
    review_cols
].sort_values(
    ["age", "height", "weight"]
)
print(records_to_review.to_string())

In [ ]:
records_to_review = records_to_review.copy()
records_to_review["bmi"] = (
    records_to_review["weight"] /
    (records_to_review["height"] / 100) ** 2
)

print(records_to_review.round(2).to_string())

In [ ]:
# Apply the cleaning function to the original data
df_clean_preview = clean_data(df_original)

In [ ]:
print("Original shape:", df_original.shape)
print("Cleaned shape:", df_clean_preview.shape)

print(
    "Rows removed:",
    len(df_original) - len(df_clean_preview)
)

In [ ]:
demographic_cols = [
    "age",
    "height",
    "weight",
    "bmi"
]
print("\nCleaned demographic ranges:")
print(
    df_clean_preview[demographic_cols]
    .agg(["min", "max"])
)

In [ ]:
print(
    "\nHeights above 250 cm:",
    (df_clean_preview["height"] > 250).sum()
)

print(
    "Infant records with height above 100 cm:",
    (
        (df_clean_preview["age"] <= 1) &
        (df_clean_preview["height"] > 100)
    ).sum()
)

print(
    "Adults with BMI below 10:",
    (
        (df_clean_preview["age"] >= 20) &
        (df_clean_preview["bmi"] < 10)
    ).sum()
)

In [ ]:
missing = df_clean_preview.isna().sum()
print("\nColumns still containing missing values:")
print(
    missing[missing > 0]
    .sort_values(ascending=False)
)
print(
    "\nDuplicate rows:",
    df_clean_preview.duplicated().sum()
)

In [ ]:
print(
    "\nj_angle still present:",
    "j_angle" in df_clean_preview.columns
)
print(
    "BMI column present:",
    "bmi" in df_clean_preview.columns
)
print(
    "arrhythmia_present column present:",
    "arrhythmia_present" in df_clean_preview.columns
)

In [ ]:
print("\nFirst five cleaned records:")
print(
    df_clean_preview[
        [
            "age",
            "sex",
            "height",
            "weight",
            "bmi",
            "heart_rate",
            "class",
            "arrhythmia_present"
        ]
    ].head()
)

In [ ]:
# Use the cleaned data already reviewed above
df_etl_clean = df_clean_preview.copy(deep=True)

with sqlite3.connect("..\\data\\etl.db") as conn:
    df_etl_clean.to_sql(
        "clean",
        conn,
        if_exists="replace",
        index=False
    )

print("ETL completed.")
print("Cleaned table shape:", df_etl_clean.shape)

In [ ]:
with sqlite3.connect("..\\data\\elt.db") as conn:
    # Load the untouched raw data first
    df_original.to_sql(
        "raw",
        conn,
        if_exists="replace",
        index=False
    )
    # Read the raw table back into Pandas
    df_elt_raw = pd.read_sql(
        "SELECT * FROM raw",
        conn
    )
    # Transform the data after loading
    df_elt_clean = clean_data(df_elt_raw)
    # Store the cleaned result as a second table
    df_elt_clean.to_sql(
        "clean",
        conn,
        if_exists="replace",
        index=False
    )

print("ELT completed.")
print("Raw table shape:", df_elt_raw.shape)
print("Cleaned table shape:", df_elt_clean.shape)

In [ ]:
with sqlite3.connect("..\\data\\etl.db") as conn:
    etl_tables = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table'",
        conn
    )
print("Tables in etl.db:")
print(etl_tables)

with sqlite3.connect("..\\data\\elt.db") as conn:
    elt_tables = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table'",
        conn
    )
print("\nTables in elt.db:")
print(elt_tables)

In [ ]:
print(df_original["heart_rate"].mean())          # one number, a Series method
print(df_original[["heart_rate", "age"]].mean())    # one number PER column, a DataFrame   

In [ ]:
# Slow method: loop
start = time.time()
slow_result = []
for i in range(len(df_original)):
    slow_result.append(
df_original["weight"].iloc[i] / df_original["height"].iloc[i]
    )
slow_time = time.time() - start

# Fast method: vectorised calculation
start = time.time()
fast_result = df_original["weight"] / df_original["height"]
fast_time = time.time() - start

# Compare the first five results
print("Slow result:")
print(slow_result[:5])
print("\nFast result:")
print(fast_result.head().tolist())

# Compare the speed
print("\nSlow time:", slow_time)
print("Fast time:", fast_time)
print("\nFast method is", slow_time / fast_time, "times faster")

In [ ]:
print(df_original.shape)
print(df_original.dtypes.value_counts())
print(df_original.info())
print(df_original.describe())

In [ ]:
summary_cols = ["age", "height", "weight", "heart_rate"]
print(df_etl_clean[summary_cols].describe())

skew_results = df_etl_clean[summary_cols].skew().round(2)
print(skew_results)
print(df_etl_clean["arrhythmia_present"].value_counts(normalize=True))

# Expected under the current rules: age and weight keep their raw minima.
print("Minimum age retained:", df_etl_clean["age"].min())
print("Minimum weight retained:", df_etl_clean["weight"].min())

In [ ]:
plt.hist(df_etl_clean["heart_rate"].dropna(), bins=20, edgecolor="white", linewidth=1)
plt.xlabel("Heart rate"); plt.ylabel("Number of patients")
plt.title("Distribution of heart rate")
plt.show()

In [ ]:
plt.hist(df_etl_clean["age"].dropna(), bins=25, edgecolor="white", linewidth=1)
plt.xlabel("Age")
plt.ylabel("Number of patients")
plt.title("Distribution of age")
plt.show()

In [ ]:
plt.hist(df_original["height"].dropna(), bins=30, edgecolor="white", linewidth=1)
plt.title("Height (before cleaning)")
plt.xlabel("Height (cm)")
plt.ylabel("Number of patients")
plt.show()
plt.hist(df_original["weight"].dropna(), bins=30, edgecolor="white", linewidth=1)
plt.title("Weight (before cleaning)")
plt.xlabel("Weight (kg)")
plt.ylabel("Number of patients")
plt.show()
plt.hist(df_original["heart_rate"].dropna(), bins=30, edgecolor="white", linewidth=1)
plt.title("Heart rate (before cleaning)")
plt.xlabel("Heart rate (bpm)")
plt.ylabel("Number of patients")
plt.show()

In [ ]:
# Compare the variables that were reviewed during cleaning
for col in ["age", "height", "weight", "heart_rate"]:
    print(
        col,
        "raw range:",
        (df_original[col].min(), df_original[col].max()),
        "clean range:",
        (df_etl_clean[col].min(), df_etl_clean[col].max())
    )

In [ ]:
plt.hist(df_etl_clean["height"].dropna(), bins=30, edgecolor="white", linewidth=1)
plt.title("Height (after cleaning)")
plt.xlabel("Height (cm)")
plt.ylabel("Number of patients")
plt.show()
plt.hist(df_etl_clean["weight"].dropna(), bins=30, edgecolor="white", linewidth=1)
plt.title("Weight (after cleaning)")
plt.xlabel("Weight (kg)")
plt.ylabel("Number of patients")
plt.show()
plt.hist(df_etl_clean["heart_rate"].dropna(), bins=30, edgecolor="white", linewidth=1)
plt.title("Heart rate (after cleaning)")
plt.xlabel("Heart rate (bpm)")
plt.ylabel("Number of patients")
plt.show()

In [ ]:
plt.boxplot(df_etl_clean["weight"].dropna(), orientation="horizontal", tick_labels=["Weight"])
plt.xlabel("Weight (kg)")
plt.show()          # Figure 10 is what you'll see

In [ ]:
def iqr_flag(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)

age_flag = iqr_flag(df_etl_clean["age"])
height_flag = iqr_flag(df_etl_clean["height"])
weight_flag = iqr_flag(df_etl_clean["weight"])
demographic_flags = age_flag | height_flag | weight_flag

print(
    df_etl_clean.loc[
        demographic_flags,
        ["age", "sex", "height", "weight", "class"]
    ].sort_values(["age", "height", "weight"]).to_string()
)

# Heart rate is checked separately with a z-score
mean_hr = df_etl_clean["heart_rate"].mean()
std_hr = df_etl_clean["heart_rate"].std()
z_hr = (df_etl_clean["heart_rate"] - mean_hr) / std_hr

print("Heart-rate values with |z| > 3:")
print(df_etl_clean.loc[z_hr.abs() > 3, ["age", "sex", "heart_rate", "class"]])

In [ ]:
print(
    df_etl_clean.loc[
        (df_etl_clean["age"] == 75) &
        (df_etl_clean["sex"] == 0) &
        (df_etl_clean["height"] == 190) &
        (df_etl_clean["weight"] == 80)
    ].to_string()
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].boxplot(df_etl_clean["height"].dropna(), tick_labels=["Height"])
axes[0].set_title("Height"); axes[0].set_ylabel("cm")
axes[1].boxplot(df_etl_clean["heart_rate"].dropna(), tick_labels=["Heart rate"])
axes[1].set_title("Heart rate"); axes[1].set_ylabel("bpm")
axes[2].boxplot(df_etl_clean["weight"].dropna(), tick_labels=["Weight"])
axes[2].set_title("Weight"); axes[2].set_ylabel("kg")
plt.suptitle("Cleaned data before optional capping")
plt.show()

In [ ]:
df_sensitivity = df_etl_clean.copy(deep=True)
df_sensitivity["height_capped"] = df_sensitivity["height"].clip(
    lower=df_sensitivity["height"].quantile(0.01)
)
df_sensitivity["heart_rate_capped"] = df_sensitivity["heart_rate"].clip(
    lower=df_sensitivity["heart_rate"].quantile(0.01),
    upper=df_sensitivity["heart_rate"].quantile(0.99)
)
df_sensitivity["weight_capped"] = df_sensitivity["weight"].clip(
    upper=df_sensitivity["weight"].quantile(0.99)
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].boxplot(df_sensitivity["height_capped"].dropna(), tick_labels=["Height"])
axes[0].set_title("Height (sensitivity-capped)"); axes[0].set_ylabel("cm")
axes[1].boxplot(df_sensitivity["heart_rate_capped"].dropna(), tick_labels=["Heart rate"])
axes[1].set_title("Heart rate (capped)"); axes[1].set_ylabel("bpm")
axes[2].boxplot(df_sensitivity["weight_capped"].dropna(), tick_labels=["Weight"])
axes[2].set_title("Weight (capped)"); axes[2].set_ylabel("kg")
plt.suptitle("Optional sensitivity-only capped view")
plt.show()

In [ ]:
rate = df_etl_clean.groupby("sex", observed=False)["arrhythmia_present"].mean()
print(rate)
sex_labels = {0: "Male", 1: "Female"}
labels = [sex_labels[int(value)] for value in rate.index]
plt.bar(labels, rate.values)
plt.xlabel("Sex")
plt.ylabel("Proportion with arrhythmia")
plt.title("Arrhythmia rate by sex")
plt.show()          # Figure 13 is what you'll see

In [ ]:
import numpy as np

num_cols = ["age", "height", "weight", "qrs_duration", "pr_interval", "qt_interval",
    "heart_rate"]
corr_matrix = df_etl_clean[num_cols].corr()
print(corr_matrix.round(2))

# Find the strongest non-diagonal linear association
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)
strongest_pair = upper_triangle.abs().stack().idxmax()
print(
    "Strongest absolute correlation:",
    strongest_pair,
    round(corr_matrix.loc[strongest_pair[0], strongest_pair[1]], 2)
)
plt.imshow(corr_matrix, cmap="RdYlGn", vmin=-1, vmax=1)
plt.xticks(range(len(num_cols)), num_cols, rotation=45, ha="right")
plt.yticks(range(len(num_cols)), num_cols)
plt.colorbar(label="Correlation")
plt.title("Correlation between 7 of 279 input features")
plt.show()          # Figure 14 is what you'll see

In [ ]:
class_names = {
    1: "Normal", 2: "Ischemic changes (CAD)", 3: "Old Ant. MI", 4: "Old Inf. MI",
    5: "Sinus tachycardia", 6: "Sinus bradycardia", 7: "PVC", 8: "Supraventricular PC",
    9: "Left bundle branch block", 10: "Right bundle branch block",
    11: "1st degree AV block", 12: "2nd degree AV block", 13: "3rd degree AV block",
    14: "Left vent. hypertrophy", 15: "Atrial fib./flutter", 16: "Other",
}
pqrst_cols = ["p_interval", "qrs_duration", "pr_interval", "qt_interval", "t_interval",
    "heart_rate"]
class_counts = df_etl_clean["class"].value_counts().sort_index()
print("Patients per class:")
print(class_counts)
profile = df_etl_clean.groupby("class")[pqrst_cols].mean()
print("Mean profile per class:")
print(profile.round(1))
# standardise each column so differing scales (ms vs bpm) do not dominate the color map
profile_z = (profile - profile.mean()) / profile.std()
row_labels = [class_names[int(c)] for c in profile_z.index]
plt.imshow(profile_z.values, cmap="RdYlGn", vmin=-2, vmax=2, aspect="auto")
plt.xticks(range(len(pqrst_cols)), pqrst_cols, rotation=45, ha="right")
plt.yticks(range(len(profile_z)), row_labels)
plt.colorbar(label="Standardised mean (per column)")
plt.title("PQRST + heart rate profile, by arrhythmia type")
plt.show()

In [ ]:
miss = df_original.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0] * 100
plt.barh(miss.index.astype(str), miss.values)
plt.xlabel("% missing")
plt.title("Missing values in the raw data")
plt.show()

In [ ]:
order = sorted(df_etl_clean["class"].unique())
x_pos = df_etl_clean["class"].map({c: i for i, c in enumerate(order)})
plt.scatter(x_pos, df_etl_clean["heart_rate"], alpha=0.4)
means = df_etl_clean.groupby("class")["heart_rate"].mean()
print("Mean heart rate by class:")
print(means.sort_values())
plt.scatter(range(len(order)), [means[c] for c in order], color="red", marker="D",
    label="Mean per type")
plt.xticks(range(len(order)), [class_names[c] for c in order], rotation=45, ha="right")
plt.xlabel("Arrhythmia type"); plt.ylabel("Heart rate (bpm)")
plt.title("Heart rate by arrhythmia type")
plt.legend()
plt.show()

In [ ]:
from matplotlib.lines import Line2D

colors = df_etl_clean["arrhythmia_present"].map({True: "orange", False: "teal"})
plt.scatter(df_etl_clean["height"], df_etl_clean["weight"], c=colors, alpha=0.5)
plt.xlabel("Height (cm)")
plt.ylabel("Weight (kg)")
plt.title("Height vs weight, by diagnosis")
legend_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="teal", markersize=8, label="No arrhythmia"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="orange", markersize=8, label="Arrhythmia present"),
]
plt.legend(handles=legend_handles)
plt.show()

In [ ]:
import numpy as np
jitter = df_etl_clean["arrhythmia_present"].astype(int) + np.random.uniform(-0.08, 0.08,
    len(df_etl_clean))
colors = df_etl_clean["arrhythmia_present"].map({True: "orange", False: "teal"})
plt.scatter(df_etl_clean["age"], jitter, c=colors, alpha=0.5)
plt.yticks([0, 1], ["No", "Yes"])
plt.xlabel("Age")
plt.ylabel("arrhythmia_present")
plt.title("Age vs arrhythmia_present")
plt.show()          # y-axis labels (No/Yes) already identify the two colours here

In [ ]:
colors = df_etl_clean["arrhythmia_present"].map({True: "orange", False: "teal"})
cols = ["height", "weight", "qt_interval", "heart_rate"]
fig, axes = plt.subplots(4, 4, figsize=(9, 9))
for i, c1 in enumerate(cols):
    for j, c2 in enumerate(cols):
        ax = axes[i, j]
        if i == j:
            ax.hist(df_etl_clean[c1].dropna(), bins=20)
        else:
            ax.scatter(df_etl_clean[c2], df_etl_clean[c1], s=4, alpha=0.35, c=colors)
        if i == 3:
            ax.set_xlabel(c2)
        if j == 0:
            ax.set_ylabel(c1)
plt.tight_layout()
from matplotlib.patches import Patch
fig.legend(handles=[
    Patch(color="teal", label="No arrhythmia"),
    Patch(color="orange", label="Arrhythmia present"),
], loc="upper right")
plt.show()

In [ ]:
from scipy import stats

a = df_etl_clean.loc[df_etl_clean["arrhythmia_present"], "heart_rate"]
b = df_etl_clean.loc[~df_etl_clean["arrhythmia_present"], "heart_rate"]
t, p = stats.ttest_ind(a, b, equal_var=False)

print("Mean with arrhythmia:", a.mean())
print("Mean without arrhythmia:", b.mean())

print(f"t = {t:.2f}, p = {p:.4f}")

if p < 0.05:
    print("Reject H0: the mean heart rates differ significantly.")
else:
    print("Fail to reject H0: no significant mean difference was detected.")

In [ ]:
table = pd.crosstab(
    df_etl_clean["sex"],
    df_etl_clean["arrhythmia_present"]
)
print(table)
chi2, p, dof, expected = stats.chi2_contingency(table)
print(f"chi2 = {chi2:.2f}, p = {p:.6f}")
if p < 0.05:
    print("Reject H0: sex and diagnosis are associated in this sample.")
else:
    print("Fail to reject H0: no significant association was detected.")